In [2]:
# CELL 1 - Mount & Setup
from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = '/content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/Turn-in-Folder' #@@@ Change to TA directory
os.makedirs(save_dir, exist_ok=True)
os.chdir(save_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# CELL 2 - Install Latest Unsloth + Zoo (fixes unsloth_zoo error + xformers fallback)
!pip install -q "git+https://github.com/unslothai/unsloth.git"
!pip install -q "git+https://github.com/unslothai/unsloth-zoo.git"

# Standard deps (no xformers needed — Unsloth will use PyTorch attn if broken)
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes -q
!pip install datasets transformers torch sacrebleu sentencepiece -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
# CELL 3 - Imports & Config
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset, concatenate_datasets
from trl import SFTTrainer
from peft import LoraConfig
from transformers import TrainingArguments, EarlyStoppingCallback
import pandas as pd
import random
from pathlib import Path

SEED = 3
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# #LOCAL MODEL AND DATA PATHS - UNCOMMENT IF YOU WANT TO RUN EVERYTHING
# DATA_DIR = Path("/content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/splits")
# MODEL_DIR = Path("/content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models")
# MODEL_DIR.mkdir(exist_ok=True)

# #LOCAL LLAMA PATH - CHANGE FOR GRADING
# local_model_path = "/content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/Llama3.2-3B"

max_seq_length = 2048  # Llama-3.2 supports 128k, we use 4k for speed/quality balance

#UNCOMMENT IF YOU WANT TO RUN EVERYTHING
# def load_base_model():
#     model, tokenizer = FastLanguageModel.from_pretrained(
#         model_name = "unsloth/Llama-3.2-3B-Instruct",
#         max_seq_length = max_seq_length,
#         dtype = None,
#         load_in_4bit = True,
#     )
#     return model

print("load_base_model() defined and ready")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
load_base_model() defined and ready


#**QLoRA, Continued Pretraining, SFT Fxn**

In [ ]:
# CELL 4 - Load Data
print("Loading datasets...")

old_geo = load_dataset("text", data_files=str(DATA_DIR / "train_old_geo.txt"), split="train")
middle_geo = load_dataset("text", data_files=str(DATA_DIR / "train_middle_geo.txt"), split="train")
modern_geo_train = load_dataset("text", data_files=str(DATA_DIR / "train_modern_geo.txt"), split="train")
modern_geo_val = load_dataset("text", data_files=str(DATA_DIR / "val_mlm_modern_geo.txt"), split="train")

paired = load_dataset("json", data_files={
    "train": str(DATA_DIR / "train_paired.jsonl"),
    "val": str(DATA_DIR / "val_paired.jsonl"),
    "test": str(DATA_DIR / "test_paired.jsonl")
})

print(f"Old: {len(old_geo):,} | Middle: {len(middle_geo):,} | Modern Train: {len(modern_geo_train):,}")
print(f"Paired: {len(paired['train']):,} train | {len(paired['val']):,} val | {len(paired['test']):,} test")

Loading datasets...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

KeyboardInterrupt: 

In [ ]:
# CELL 5 - Load Official Unsloth Llama-3.2-3B 4-bit (fastest, no local issues)
model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Llama-3.2-3B-bnb-4bit",   # Official, public, instant load
    max_seq_length = max_seq_length,
    dtype = None,                     # Auto → bfloat16 on A100
    load_in_4bit = True,
)

# Fix padding
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✅ Successfully loaded unsloth/Llama-3.2-3B-bnb-4bit")
print(f"   Vocab size: {len(tokenizer)} | 4-bit: {model.is_loaded_in_4bit}")

In [ ]:
# CELL 6 - PERMANENT FIX: Use the official Unsloth Llama-3.2-3B 4-bit model (instant, no config errors)
model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Llama-3.2-3B-bnb-4bit",      # Official, always works, 4-bit, blazing fast
    max_seq_length = max_seq_length,
    dtype = None,                        # Auto → bfloat16 on A100
    load_in_4bit = True,
)

# Fix padding token
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✅ PERFECTLY LOADED unsloth/Llama-3.2-3B-bnb-4bit")
print(f"   Vocab size: {len(tokenizer)} | Model type: {model.config.model_type}")

==((====))==  Unsloth 2025.12.5: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ PERFECTLY LOADED unsloth/Llama-3.2-3B-bnb-4bit
   Vocab size: 128256 | Model type: llama


# **Training Variants**

In [ ]:
# CELL 7 - LoRA Config (same as before)
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none",
    task_type="SEQ_2_SEQ_LM"
)

In [ ]:
# CELL 8 - Resume Helper
def get_latest_checkpoint(output_dir):
    dir_path = Path(output_dir)
    if not dir_path.exists():
        return None
    checkpoints = list(dir_path.glob("checkpoint-*"))
    if not checkpoints:
        return None
    return str(max(checkpoints, key=lambda x: int(x.name.split("-")[-1])))

In [ ]:
# === FINAL PATCH: Run ONCE after restart, NEVER again ===
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "1"

from transformers import TrainingArguments

# Save the TRUE original __init__ (before any patch)
if not hasattr(TrainingArguments, "_unsloth_original_init"):
    TrainingArguments._unsloth_original_init = TrainingArguments.__init__
    print("Saved original TrainingArguments.__init__")

def _safe_training_args_init(self, *args, **kwargs):
    # Force safe values
    kwargs["dataloader_num_workers"] = 0
    kwargs["dataloader_pin_memory"] = False
    # Call the REAL original
    return TrainingArguments._unsloth_original_init(self, *args, **kwargs)

# Apply patch ONLY if not already patched
if TrainingArguments.__init__ is not _safe_training_args_init:
    TrainingArguments.__init__ = _safe_training_args_init
    print("TrainingArguments patched: dataloader_num_workers=0, pin_memory=False")
else:
    print("Patch already applied — skipping")

Saved original TrainingArguments.__init__
TrainingArguments patched: dataloader_num_workers=0, pin_memory=False


In [ ]:
# CELL 9 - FINAL MAX-SPEED VERSION FOR A100 (NO PACKING, WORKS 100%)
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling, EarlyStoppingCallback

def mlm_pretrain(model, train_datasets, val_dataset, output_dir, resume=False):
    if isinstance(train_datasets, list):
        train_dataset = concatenate_datasets(train_datasets).shuffle(seed=SEED)
    else:
        train_dataset = train_datasets

    # Keep truncation=True + max_length=2048 (fastest stable config)
    def tokenize_function(examples):
        return tokenizer(examples["text"], truncation=True, max_length=2048)

    tokenized_train = train_dataset.map(tokenize_function, batched=True, num_proc=1,
                                       remove_columns=train_dataset.column_names)
    tokenized_val   = val_dataset.map(tokenize_function, batched=True, num_proc=1,
                                      remove_columns=val_dataset.column_names)

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            per_device_train_batch_size=32,       # ← MAX possible on A100 80GB with 4-bit + bf16
            gradient_accumulation_steps=1,
            learning_rate=1e-5,
            num_train_epochs=3,
            weight_decay=0.01,
            bf16=True,
            fp16=False,
            logging_steps=50,
            save_steps=5000,                      # Save less often
            eval_strategy="steps",
            eval_steps=1000,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            save_total_limit=2,
            output_dir=output_dir,
            report_to="none",
            seed=SEED,
            resume_from_checkpoint=resume if resume else None,
            prediction_loss_only=True,
            dataloader_num_workers=4,             # ← A100 can handle this
            dataloader_pin_memory=True,
            gradient_checkpointing="unsloth",
            optim="adamw_torch_fused",
            torch_compile=False,
            group_by_length=True,                 # ← Helps a lot with variable-length Georgian
            warmup_steps=200,
            remove_unused_columns=False,
        ),
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        data_collator=data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)],
    )
    trainer.train(resume_from_checkpoint=resume if resume else None)
    model.save_pretrained(output_dir + "_final")
    return model

In [ ]:
# CELL 10 - SFT Fine-tuning (supports custom TrainingArguments + current formatting)
from transformers import EarlyStoppingCallback, TrainingArguments
from trl import SFTTrainer

def sft_finetune(model, train_dataset, val_dataset, output_dir, resume=False, baseline=False, training_args=None):
    # Your current formatting (simple and effective for Georgian)
    def formatting_func(example):
        text = f"Translate English to Georgian:\n{example['en']}\n###\n{example['ka']}"
        return [text]

    # Epochs: 8 for baseline, 3 for others
    epochs = 5

    # If no custom args provided, use defaults (matching your original Cell 10)
    if training_args is None:
        training_args = TrainingArguments(
            per_device_train_batch_size=16,
            gradient_accumulation_steps=1,
            warmup_steps=5,
            learning_rate=2e-4,
            num_train_epochs=epochs,
            bf16=True,
            fp16=False,
            logging_steps=1,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=SEED,
            output_dir=output_dir,
            report_to="none",
            save_strategy="steps",
            save_steps=200,
            eval_strategy="steps",
            eval_steps=200,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            save_total_limit=2,
            resume_from_checkpoint=resume,
        )
    else:
        # Apply key settings when custom args are passed
        training_args.num_train_epochs = epochs
        training_args.output_dir = output_dir
        training_args.resume_from_checkpoint = resume

    # Create trainer
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        formatting_func=formatting_func,
        max_seq_length=max_seq_length,
        dataset_num_proc=1,
        packing=False,
        args=training_args,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)],
    )

    trainer.train(resume_from_checkpoint=resume)
    model.save_pretrained(output_dir + "_final")
    return model

In [ ]:
# CELL 11 - 1. Baseline (no historical pretraining) → 8 epochs SFT
print("=== TRAINING BASELINE (8 epochs SFT) ===")
model = load_base_model()

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=32,
    lora_dropout=0.0,  # Changed to 0.0 to avoid performance hit
    bias="none",
    use_gradient_checkpointing="unsloth",  # Corrected from "unslosh"
    random_state=SEED,
    max_seq_length=max_seq_length,
)

# Supervised Fine-Tuning
model = sft_finetune(model, paired["train"], paired["val"],
                     f"{MODEL_DIR}/baseline_sft", resume=False, baseline=True)

print("Merging LoRA adapters into the base model...")
model = model.merge_and_unload()  # Merge LoRA adapters into base model

# Save the full merged model
merged_save_path = f"{MODEL_DIR}/baseline_final_merged"
model.save_pretrained(merged_save_path)
tokenizer.save_pretrained(merged_save_path)
print(f"Full merged baseline model saved to {merged_save_path}")

# Optional: Save LoRA-only checkpoint for reproducibility
lora_save_path = f"{MODEL_DIR}/baseline_final_lora_only"
model.save_pretrained(lora_save_path, save_adapter=True)
print(f"LoRA-only checkpoint saved to {lora_save_path}")

# Clean up
del model
torch.cuda.empty_cache()
import gc; gc.collect()

=== TRAINING BASELINE (8 epochs SFT) ===
==((====))==  Unsloth 2025.12.5: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Map (num_proc=1):   0%|          | 0/144372 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/5000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 145 | Num Epochs = 50 | Total steps = 500
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 9,175,040 of 3,221,924,864 (0.28% trained)


Step,Training Loss,Validation Loss
200,1.658100,2.622535
400,1.058000,3.393031


Merging LoRA adapters into the base model...


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Full merged baseline model saved to /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/baseline_final_merged
LoRA-only checkpoint saved to /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/baseline_final_lora_only


9426

In [ ]:
# CELL - SFT Fine-Tuning for Continued-Pretrained Models (mixed, chron, rev)
# FINAL WORKING VERSION: No LoRA (train full model) + your improved hyperparameters + bf16

from unsloth import FastLanguageModel
from transformers import TrainingArguments
import torch
import gc

print("=== SFT FINE-TUNING FOR CONTINUED-PRETRAINED MODELS (mixed, chron, rev) ===")

mlm_checkpoints = {
    "mixed": f"{MODEL_DIR}/mixed_mlm_final",
    "chron": f"{MODEL_DIR}/chron_mlm_final",
    "rev":   f"{MODEL_DIR}/rev_mlm_final",
}

# Use global max_seq_length
max_seq_length_sft = max_seq_length

# Optimized hyperparameters
sft_epochs = 50
learning_rate = 1e-5
per_device_batch_size = 8
gradient_accum_steps = 2

for name, mlm_path in mlm_checkpoints.items():
    print(f"\n--- Starting SFT for {name.upper()} model ---")
    print(f"Loading continued-pretrained checkpoint from: {mlm_path}")

    # Load the continued-pretrained model (full, not LoRA)
    model, _ = FastLanguageModel.from_pretrained(
        mlm_path,
        max_seq_length=max_seq_length_sft,
        dtype=None,                # Unsloth chooses bfloat16 on A100
        load_in_4bit=True,
    )

    # IMPORTANT: Train the full model (no LoRA) to avoid grad_fn issues
    # This is simpler and stable for continued pretraining → SFT
    print("Model loaded — training full parameters with improved settings (no LoRA).")

    # Output directory
    sft_output_dir = f"{MODEL_DIR}/{name}_sft"

    # Custom training args
    custom_training_args = TrainingArguments(
        per_device_train_batch_size=per_device_batch_size,
        gradient_accumulation_steps=gradient_accum_steps,
        warmup_steps=5,
        learning_rate=learning_rate,
        num_train_epochs=sft_epochs,
        bf16=True,
        fp16=False,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        output_dir=sft_output_dir,
        report_to="none",
        save_strategy="steps",
        save_steps=200,
        eval_strategy="steps",
        eval_steps=200,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        gradient_checkpointing=True,
    )

    effective_batch = per_device_batch_size * gradient_accum_steps
    print(f"Starting SFT ({sft_epochs} epochs, lr={learning_rate}, effective batch={effective_batch}, max_seq_length={max_seq_length_sft}, bf16 enabled)...")

    # Run SFT on the full model
    model = sft_finetune(
        model=model,
        train_dataset=paired["train"],
        val_dataset=paired["val"],
        output_dir=sft_output_dir,
        resume=False,
        baseline=False,
        training_args=custom_training_args
    )

    print(f"{name.upper()} SFT completed!")

    # Save final checkpoint (full model)
    final_checkpoint = f"{sft_output_dir}_final"
    model.save_pretrained(final_checkpoint)
    tokenizer.save_pretrained(final_checkpoint)
    print(f"Final checkpoint saved: {final_checkpoint}")

    # Cleanup
    del model
    torch.cuda.empty_cache()
    gc.collect()

    print(f"--- {name.upper()} SFT FINISHED ---\n")

print("=== ALL CONTINUED-PRETRAINING MODELS SUCCESSFULLY FINE-TUNED ===")
print("Your four final models are ready for evaluation:")
print(f"  • Baseline (direct SFT):                 {MODEL_DIR}/baseline_sft_final")
print(f"  • Mixed MLM + SFT:                       {MODEL_DIR}/mixed_sft_final")
print(f"  • Chronological MLM + SFT:               {MODEL_DIR}/chron_sft_final")
print(f"  • Reverse-Chronological MLM + SFT:       {MODEL_DIR}/rev_sft_final")
print("\nYou can now run your 50-sample evaluation to test your curriculum hypothesis for English → Modern Georgian translation.")

=== SFT FINE-TUNING FOR CONTINUED-PRETRAINED MODELS (mixed, chron, rev) ===

--- Starting SFT for MIXED model ---
Loading continued-pretrained checkpoint from: /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/mixed_mlm_final
==((====))==  Unsloth 2025.12.5: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model loaded — training full parameters with improved settings (no LoRA).
Starting SFT (50 epochs, lr=1e-05, effective batch=16, max_seq_length=2048, bf16 enabled)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 145 | Num Epochs = 50 | Total steps = 500
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 9,175,040 of 3,221,924,864 (0.28% trained)


Step,Training Loss,Validation Loss
200,2.499900,2.506558
400,2.436600,2.469966


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


MIXED SFT completed!
Final checkpoint saved: /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/mixed_sft_final
--- MIXED SFT FINISHED ---


--- Starting SFT for CHRON model ---
Loading continued-pretrained checkpoint from: /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/chron_mlm_final
==((====))==  Unsloth 2025.12.5: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model loaded — training full parameters with improved settings (no LoRA).
Starting SFT (50 epochs, lr=1e-05, effective batch=16, max_seq_length=2048, bf16 enabled)...


...

Step,Training Loss,Validation Loss
200,2.500900,2.507613
400,2.437200,2.470806


CHRON SFT completed!
Final checkpoint saved: /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/chron_sft_final
--- CHRON SFT FINISHED ---


--- Starting SFT for REV model ---
Loading continued-pretrained checkpoint from: /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/rev_mlm_final
==((====))==  Unsloth 2025.12.5: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model loaded — training full parameters with improved settings (no LoRA).
Starting SFT (50 epochs, lr=1e-05, effective batch=16, max_seq_length=2048, bf16 enabled)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 145 | Num Epochs = 50 | Total steps = 500
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 9,175,040 of 3,221,924,864 (0.28% trained)


Step,Training Loss,Validation Loss
200,2.498600,2.503329
400,2.435500,2.466457


REV SFT completed!
Final checkpoint saved: /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/rev_sft_final
--- REV SFT FINISHED ---

=== ALL CONTINUED-PRETRAINING MODELS SUCCESSFULLY FINE-TUNED ===
Your four final models are ready for evaluation:
  • Baseline (direct SFT):                 /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/baseline_sft_final
  • Mixed MLM + SFT:                       /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/mixed_sft_final
  • Chronological MLM + SFT:               /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/chron_sft_final
  • Reverse-Chronological MLM + SFT:       /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/rev_sft_final

You can now run your 50-sample evaluation to test your curriculum hypothesis for English → Modern Georgian translation.


In [ ]:
# DEBUGGING CELL: Test ALL 4 models on the SAME SINGLE English sentence
# Compares baseline, mixed, chronological, and reverse models side-by-side

import json
import re
import torch
import gc
from datasets import Dataset
from unsloth import FastLanguageModel
from peft import PeftModel
from transformers import StoppingCriteria, StoppingCriteriaList

# List of all 4 models in order
models_to_test = [
    {"name": "baseline", "display": "Baseline (direct SFT)"},
    {"name": "mixed",    "display": "Mixed MLM + SFT"},
    {"name": "chron",    "display": "Chronological MLM + SFT"},
    {"name": "rev",      "display": "Reverse-Chronological MLM + SFT"},
]

# Single English sentence to translate (choose a representative one)
# You can change this to any sentence from your test set
test_sentence = "For God so loved the world, that he gave his only begotten Son, that whosoever believeth in him should not perish, but have everlasting life."

# Load reference Georgian translation (optional - for comparison)
# Adjust path if needed
val_path = "/content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/splits/test_paired.jsonl"
val_lines = [json.loads(line) for line in open(val_path, "r", encoding="utf-8")]
val_dataset = Dataset.from_list(val_lines)

# Find a matching reference (if available) - otherwise leave blank
reference = ""
for ex in val_dataset:
    if ex["en"].strip() == test_sentence.strip():
        reference = ex["ka"]
        break

# Exact prompt template used during training (matches your current Cell 10)
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Translate the following English text to Georgian.

### Input:
{en}

### Response:
"""

# Stopping criteria: stop at first sentence-ending punctuation
class StopOnPunctuation(StoppingCriteria):
    def __call__(self, input_ids, scores, **kwargs):
        decoded = tokenizer.decode(input_ids[0], skip_special_tokens=True)
        return any(decoded.endswith(p) for p in [".", "!", "?", "჻", "\n\n", "###"])

print("="*100)
print("DEBUGGING ALL 4 MODELS ON ONE ENGLISH SENTENCE")
print("="*100)
print(f"ENGLISH INPUT:\n{test_sentence}\n")
if reference:
    print(f"REFERENCE (Georgian):\n{reference}\n")
else:
    print("REFERENCE (Georgian): Not found in test set\n")
print("="*100)

# Test each model
for idx, model_info in enumerate(models_to_test):
    name = model_info["name"]
    display_name = model_info["display"]
    checkpoint_path = f"{MODEL_DIR}/{name}_sft_final"

    print(f"Loading {display_name} from: {checkpoint_path}")

    # Load base model
    model, tokenizer = FastLanguageModel.from_pretrained(
        "unsloth/Llama-3.2-3B-Instruct",
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )

    # Load LoRA adapters
    model = PeftModel.from_pretrained(model, checkpoint_path)
    FastLanguageModel.for_inference(model)

    # Format prompt
    prompt = alpaca_prompt.format(en=test_sentence)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=384,
        temperature=0.0,
        do_sample=False,
        use_cache=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        stopping_criteria=StoppingCriteriaList([StopOnPunctuation()]),
    )

    raw = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract translation
    if "### Response:" in raw:
        translation = raw.split("### Response:")[-1].strip()
    else:
        translation = raw[len(prompt):].strip()

    # Keep only first sentence
    translation = re.split(r'[.!?჻]', translation)[0].strip()
    if translation and translation[-1] not in ".!?჻":
        translation += "."

    print(f"\nPREDICTION ({display_name}):\n{translation}")
    print("-" * 100)

    # Cleanup
    del model, inputs, output
    torch.cuda.empty_cache()
    gc.collect()

print("\nDEBUG COMPLETE — ALL 4 MODELS TESTED ON THE SAME SENTENCE")
print("\nCompare the Georgian outputs above to evaluate:")
print("• Which curriculum (chronological vs reverse vs mixed vs none) produces the most accurate, fluent modern Georgian?")
print("• Does historical exposure help or hurt translation to modern Georgian?")

DEBUGGING ALL 4 MODELS ON ONE ENGLISH SENTENCE
ENGLISH INPUT:
For God so loved the world, that he gave his only begotten Son, that whosoever believeth in him should not perish, but have everlasting life.

REFERENCE (Georgian): Not found in test set

Loading Baseline (direct SFT) from: /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/baseline_sft_final
==((====))==  Unsloth 2025.12.5: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

PREDICTION (Baseline (direct SFT)):
Naqadi amsakheleba, amsakheleba amsakheleba, amsakheleba amsakheleba, amsakheleba amsakheleba, amsakheleba amsakheleba, amsakheleb

In [ ]:
# Quick test: Check tokenizer for causal LM (no mask needed)
print(f"Pad token: '{tokenizer.pad_token}' (ID: {tokenizer.pad_token_id})")
print(f"EOS token: '{tokenizer.eos_token}' (ID: {tokenizer.eos_token_id})")
print("✅ Causal LM-ready! No mask token required. Proceed to Cell 12.")

Pad token: '<|end_of_text|>' (ID: 128001)
EOS token: '<|end_of_text|>' (ID: 128001)
✅ Causal LM-ready! No mask token required. Proceed to Cell 12.


In [ ]:
# CELL 12 — Mixed (with AUTO-RESUME from latest checkpoint)
print("=== TRAINING MIXED (3 epochs causal pretrain → 3 epochs SFT) ===")


resume_checkpoint = get_latest_checkpoint(f"{MODEL_DIR}/mixed_mlm")  # ← your helper function from earlier
if resume_checkpoint:
    print(f"Resuming from checkpoint: {resume_checkpoint}")
else:
    print("No checkpoint found → starting from scratch")

model = load_base_model()
model = FastLanguageModel.get_peft_model(model,
    r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=32, lora_dropout=0.05, bias="none",
    use_gradient_checkpointing="unsloth", random_state=SEED, max_seq_length=max_seq_length)

mixed_train = concatenate_datasets([old_geo, middle_geo, modern_geo_train]).shuffle(seed=SEED)

# ← PASS THE RESUME PATH HERE
model = mlm_pretrain(model, [mixed_train], modern_geo_val,
                     f"{MODEL_DIR}/mixed_mlm",
                     resume=resume_checkpoint)   # ← THIS ENABLES RESUME

model = sft_finetune(model, paired["train"], paired["val"],
                     f"{MODEL_DIR}/mixed_sft", resume=False, baseline=False)

model.save_pretrained(f"{MODEL_DIR}/mixed_final")
print("Mixed model permanently saved!")

=== TRAINING MIXED (3 epochs causal pretrain → 3 epochs SFT) ===
Resuming from checkpoint: /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/mixed_mlm/checkpoint-15000
==((====))==  Unsloth 2025.12.4: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Map (num_proc=1):   0%|          | 0/237654 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/1000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 237,654 | Num Epochs = 3 | Total steps = 22,281
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 1 x 1) = 32
 "-____-"     Trainable parameters = 9,175,040 of 3,221,924,864 (0.28% trained)


Step,Training Loss,Validation Loss
16000,0.564600,0.536802
17000,0.568900,0.535248
18000,0.567200,0.533550
19000,0.566200,0.532210
20000,0.560800,0.529640
21000,0.559100,0.529580


...

Map (num_proc=1):   0%|          | 0/144372 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/5000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 145 | Num Epochs = 3 | Total steps = 30
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 9,175,040 of 3,221,924,864 (0.28% trained)


Step,Training Loss,Validation Loss


Mixed model permanently saved!


In [ ]:
# CELL 13 - TRUE CHRONOLOGICAL (Old → Middle → Modern) — NOW RESUMES CORRECTLY
print("=== TRAINING CHRONOLOGICAL (Old → Middle → Modern, strict order) ===")

resume_checkpoint = get_latest_checkpoint(f"{MODEL_DIR}/chron_mlm")
print(f"Chrono resume: {resume_checkpoint or 'starting fresh'}")

model = load_base_model()
model = FastLanguageModel.get_peft_model(model,
    r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=32, lora_dropout=0.05, bias="none",
    use_gradient_checkpointing="unsloth", random_state=SEED, max_seq_length=max_seq_length)

chron_train = concatenate_datasets([
    old_geo,
    middle_geo,
    modern_geo_train
])

print(f"Chronological dataset: {len(chron_train)} examples")
print("Order: Old Georgian (first) → Middle → Modern (last)")

# ← RESUME FROM CHECKPOINT
model = mlm_pretrain(model, [chron_train], modern_geo_val,
                     f"{MODEL_DIR}/chron_mlm", resume=resume_checkpoint)  # ← FIXED

model = sft_finetune(model, paired["train"], paired["val"],
                     f"{MODEL_DIR}/chron_sft", resume=False, baseline=False)
model.save_pretrained(f"{MODEL_DIR}/chron_final")

#del model; torch.cuda.empty_cache(); gc.collect()

=== TRAINING CHRONOLOGICAL (Old → Middle → Modern, strict order) ===
Chrono resume: /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/chron_mlm/checkpoint-10000
==((====))==  Unsloth 2025.12.4: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Chronological dataset: 237654 examples
Order: Old Georgian (first) → Middle → Modern (last)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 237,654 | Num Epochs = 3 | Total steps = 22,281
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 1 x 1) = 32
 "-____-"     Trainable parameters = 9,175,040 of 3,221,924,864 (0.28% trained)


Step,Training Loss,Validation Loss
11000,0.582200,0.553358
12000,0.579100,0.548549
13000,0.578600,0.544566
14000,0.572100,0.542374
15000,0.562500,0.539737
16000,0.567500,0.536753
17000,0.558300,0.535761
18000,0.560500,0.533309
19000,0.565800,0.532191
20000,0.562200,0.529468


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 145 | Num Epochs = 3 | Total steps = 30
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 9,175,040 of 3,221,924,864 (0.28% trained)


Step,Training Loss,Validation Loss


NameError: name 'gc' is not defined

In [ ]:
# CELL 14 - TRUE REVERSE CHRONOLOGICAL (Modern → Middle → Old) — NOW RESUMES
print("=== TRAINING REVERSE CHRONOLOGICAL (Modern → Middle → Old) ===")

resume_checkpoint = get_latest_checkpoint(f"{MODEL_DIR}/rev_mlm")
print(f"Reverse resume: {resume_checkpoint or 'starting fresh'}")

model = load_base_model()
model = FastLanguageModel.get_peft_model(model,
    r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=32, lora_dropout=0.05, bias="none",
    use_gradient_checkpointing="unsloth", random_state=SEED, max_seq_length=max_seq_length)

rev_train = concatenate_datasets([
    modern_geo_train,
    middle_geo,
    old_geo
])

print(f"Reverse dataset: {len(rev_train)} examples")

model = mlm_pretrain(model, [rev_train], modern_geo_val,
                     f"{MODEL_DIR}/rev_mlm", resume=resume_checkpoint)   # ← NOT False!

model = sft_finetune(model, paired["train"], paired["val"],
                     f"{MODEL_DIR}/rev_sft", resume=False, baseline=False)
model.save_pretrained(f"{MODEL_DIR}/rev_final")

#del model; torch.cuda.empty_cache(); gc.collect()

=== TRAINING REVERSE CHRONOLOGICAL (Modern → Middle → Old) ===
Reverse resume: /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/rev_mlm/checkpoint-10000
==((====))==  Unsloth 2025.12.4: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Reverse dataset: 237654 examples


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 237,654 | Num Epochs = 3 | Total steps = 22,281
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 1 x 1) = 32
 "-____-"     Trainable parameters = 9,175,040 of 3,221,924,864 (0.28% trained)


Step,Training Loss,Validation Loss
11000,0.577800,0.552879
12000,0.585800,0.548194
13000,0.574900,0.544470
14000,0.576300,0.542424
15000,0.566900,0.539425
16000,0.566200,0.537423
17000,0.567400,0.535263
18000,0.564200,0.533800
19000,0.564400,0.531817
20000,0.566700,0.529147


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 145 | Num Epochs = 3 | Total steps = 30
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 9,175,040 of 3,221,924,864 (0.28% trained)


Step,Training Loss,Validation Loss


NameError: name 'gc' is not defined

# **Merge Lora**

In [ ]:
# CELL 15.5 - FINAL WORKING MERGE CELL (tested with Unsloth 2025.12.4 + PEFT ≥0.13.2)
import torch
import os
import gc
from unsloth import FastLanguageModel

base_model_path = "unsloth/Llama-3.2-3B-Instruct"

lora_checkpoints = {
    "baseline":      f"{MODEL_DIR}/baseline_final",
    "mixed":         f"{MODEL_DIR}/mixed_final",
    "chronological": f"{MODEL_DIR}/chron_final",
    "reverse":       f"{MODEL_DIR}/rev_final",
}

print("=== Merging all LoRA models into full 3B models ===\n")

for name, lora_path in lora_checkpoints.items():
    merged_path = f"{MODEL_DIR}/{name}_final_merged"

    # Skip if already merged
    if os.path.exists(merged_path):
        print(f"✓ {name:14} → already merged at {merged_path}")
        continue

    print(f"Merging {name:14} → {merged_path}")

    # 1. Load base model (full precision)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = base_model_path,
        max_seq_length = 2048,
        dtype = None,           # bf16 on A100
        load_in_4bit = False,   # must be False for merging
    )

    # 2. Apply identical LoRA config as during training
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_alpha = 32,
        lora_dropout = 0.0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = SEED,
        max_seq_length = 2048,
    )

    # 3. Load the trained adapter — CORRECT argument name for newest PEFT
    model.load_adapter(
        model_id = lora_path,           # ← this is the fix
        adapter_name = "georgian_lora"  # can be any string
    )
    model.set_adapter("georgian_lora")

    # 4. Merge & unload
    print("   Merging LoRA into base model...")
    model = model.merge_and_unload()

    # 5. Save full merged model + tokenizer
    model.save_pretrained(merged_path)
    tokenizer.save_pretrained(merged_path)
    print(f"   Saved full merged model (~12–13 GB) → {merged_path}\n")

    # 6. Clean up
    del model, tokenizer
    torch.cuda.empty_cache()
    gc.collect()

print("=== ALL 4 MODELS ARE NOW FULLY MERGED AND READY FOR 4-BIT INFERENCE ===")
print("Update your CELL 16 paths to the *_final_merged folders and run!")

=== Merging all LoRA models into full 3B models ===

✓ baseline       → already merged at /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/baseline_final_merged
Merging mixed          → /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/mixed_final_merged
==((====))==  Unsloth 2025.12.4: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

   Merging LoRA into base model...
   Saved full merged model (~12–13 GB) → /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/mixed_final_merged

Merging chronological  → /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/chronological_final_merged
==((====))==  Unsloth 2025.12.4: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

   Merging LoRA into base model...
   Saved full merged model (~12–13 GB) → /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/chronological_final_merged

Merging reverse        → /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/reverse_final_merged
==((====))==  Unsloth 2025.12.4: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

   Merging LoRA into base model...
   Saved full merged model (~12–13 GB) → /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/models/reverse_final_merged

=== ALL 4 MODELS ARE NOW FULLY MERGED AND READY FOR 4-BIT INFERENCE ===
Update your CELL 16 paths to the *_final_merged folders and run!


# **Evaluation**

In [5]:
# FINAL EVALUATION CELL: Run a single sentence through ALL 4 models
# Saves source_en, reference_ka (empty), and predictions from all models to CSV

import json
import re
import torch
import gc
import pandas as pd
from unsloth import FastLanguageModel
from peft import PeftModel
from transformers import StoppingCriteria, StoppingCriteriaList
from IPython.display import display, HTML

print("=== SINGLE SENTENCE EVALUATION ON ALL 4 MODELS ===")

# Hardcoded single sentence to translate (English to Georgian) for TA Running @@@
src = "The Book of Mormon is true."
ref = "მორმონის წიგნი ჭეშმარიტია."

# Define all 4 models
models_to_eval = [
    {"name": "baseline", "display": "Baseline"},
    {"name": "mixed",    "display": "Mixed MLM + SFT"},
    {"name": "chron",    "display": "Chronological MLM + SFT"},
    {"name": "rev",      "display": "Reverse-Chronological MLM + SFT"},
]

# Exact prompt template used during training
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Translate the following English text to Georgian.

### Input:
{en}

### Response:
"""

# Stopping criteria: stop at first sentence-ending punctuation
class StopOnPunctuation(StoppingCriteria):
    def __call__(self, input_ids, scores, **kwargs):
        decoded = tokenizer.decode(input_ids[0], skip_special_tokens=True)
        return any(decoded.endswith(p) for p in [".", "!", "?", "჻", "\n\n", "###"])

# Results storage
results = {
    "source_en": [src],
    "reference_ka": [ref],
}
for m in models_to_eval:
    results[m["name"]] = []

print(f"Evaluating single sentence across {len(models_to_eval)} models...\n")
print(f"Processing: {src[:80]}{'...' if len(src)>80 else ''}")

for model_info in models_to_eval:
    name = model_info["name"]
    display = model_info["display"]
    checkpoint_path = f"/content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/Turn-in-Folder/{name}_sft_final" #@@@ change to TA drive directory

    # Load model
    model, tokenizer = FastLanguageModel.from_pretrained(
        "unsloth/Llama-3.2-3B-Instruct",
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )
    model = PeftModel.from_pretrained(model, checkpoint_path)
    FastLanguageModel.for_inference(model)

    # Format prompt
    prompt = alpaca_prompt.format(en=src)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=384,
        temperature=0.0,
        do_sample=False,
        use_cache=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        stopping_criteria=StoppingCriteriaList([StopOnPunctuation()]),
    )

    raw = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract translation
    if "### Response:" in raw:
        translation = raw.split("### Response:")[-1].strip()
    else:
        translation = raw[len(prompt):].strip()

    results[name].append(translation)

    # Cleanup
    del model, inputs, output
    torch.cuda.empty_cache()
    gc.collect()

print("-" * 80)

# Create DataFrame
df = pd.DataFrame(results)

# Reorder columns for clarity
columns_order = ["source_en", "reference_ka", "baseline", "mixed", "chron", "rev"]
df = df[columns_order]

# Save to CSV
output_csv = f"single_sample_evaluation_all_4_models.csv"
df.to_csv(output_csv, index=False, encoding="utf-8")
print(f"\nEVALUATION COMPLETE!")
print(f"Results saved to: {output_csv}")
print(f"\nColumns: source_en | reference_ka | baseline | mixed | chron | rev")

=== SINGLE SENTENCE EVALUATION ON ALL 4 MODELS ===
Evaluating single sentence across 4 models...

Processing: The Book of Mormon is true.
==((====))==  Unsloth 2025.12.6: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
==((====))==  Unsloth 2025.12.6: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading 

# **AUTOMATIC EVAL IF DESIRED**

In [15]:
# FINAL EVALUATION CELL: Run 50 test sentences through ALL 4 models
# Saves source_en, reference_ka, and predictions from all models to CSV

import json
import re
import torch
import gc
import pandas as pd
from datasets import Dataset
from unsloth import FastLanguageModel
from peft import PeftModel
from transformers import StoppingCriteria, StoppingCriteriaList
from IPython.display import display, HTML

print("=== 50-SAMPLE EVALUATION ON ALL 4 MODELS ===")

# Load test set - CHANGE PATH
test_path = "/content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/Turn-in-Folder/test_paired.jsonl" #@@@ change to TA Drive directory
test_lines = [json.loads(line) for line in open(test_path, "r", encoding="utf-8")]
test_dataset = Dataset.from_list(test_lines)

# Fixed seed for reproducible 50 examples
test_sample = test_dataset.shuffle(seed=42).select(range(50))

# Define all 4 models
models_to_eval = [
    {"name": "baseline", "display": "Baseline"},
    {"name": "mixed",    "display": "Mixed MLM + SFT"},
    {"name": "chron",    "display": "Chronological MLM + SFT"},
    {"name": "rev",      "display": "Reverse-Chronological MLM + SFT"},
]

# Exact prompt template used during training
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Translate the following English text to Georgian.

### Input:
{en}

### Response:
"""

# Stopping criteria: stop at first sentence-ending punctuation
class StopOnPunctuation(StoppingCriteria):
    def __call__(self, input_ids, scores, **kwargs):
        decoded = tokenizer.decode(input_ids[0], skip_special_tokens=True)
        return any(decoded.endswith(p) for p in [".", "!", "?", "჻", "\n\n", "###"])

# Results storage
results = {
    "source_en": [],
    "reference_ka": [],
}
for m in models_to_eval:
    results[m["name"]] = []

print(f"Evaluating 50 test sentences across {len(models_to_eval)} models...\n")

for idx, example in enumerate(test_sample):
    src = example["en"]
    ref = example["ka"]
    results["source_en"].append(src)
    results["reference_ka"].append(ref)

    print(f"[{idx+1:02d}/50] Processing: {src[:80]}{'...' if len(src)>80 else ''}")

    for model_info in models_to_eval:
        name = model_info["name"]
        display = model_info["display"]
        checkpoint_path = f"/content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/Turn-in-Folder/{name}_sft_final" #@@@ change to TA drive directory

        # Load model
        model, tokenizer = FastLanguageModel.from_pretrained(
            "unsloth/Llama-3.2-3B-Instruct",
            max_seq_length=2048,
            dtype=None,
            load_in_4bit=True,
        )
        model = PeftModel.from_pretrained(model, checkpoint_path)
        FastLanguageModel.for_inference(model)

        # Format prompt
        prompt = alpaca_prompt.format(en=src)

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        output = model.generate(
            **inputs,
            max_new_tokens=384,
            temperature=0.0,
            do_sample=False,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            stopping_criteria=StoppingCriteriaList([StopOnPunctuation()]),
        )

        raw = tokenizer.decode(output[0], skip_special_tokens=True)

        # Extract translation
        if "### Response:" in raw:
            translation = raw.split("### Response:")[-1].strip()
        else:
            translation = raw[len(prompt):].strip()

        results[name].append(translation)

        # Cleanup
        del model, inputs, output
        torch.cuda.empty_cache()
        gc.collect()

    print("-" * 80)

# Create DataFrame
df = pd.DataFrame(results)

# Reorder columns for clarity
columns_order = ["source_en", "reference_ka", "baseline", "mixed", "chron", "rev"]
df = df[columns_order]

# Save to CSV
output_csv = f"50_sample_evaluation_all_4_models.csv"
df.to_csv(output_csv, index=False, encoding="utf-8")
print(f"\nEVALUATION COMPLETE!")
print(f"Results saved to: {output_csv}")
print(f"\nColumns: source_en | reference_ka | baseline | mixed | chron | rev")



=== 50-SAMPLE EVALUATION ON ALL 4 MODELS ===
Evaluating 50 test sentences across 4 models...

[01/50] Processing: But behold, as the king came out to meet him Amalickiah caused that his servants...
==((====))==  Unsloth 2025.12.6: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
==((====))==  Unsloth 2025.12.6: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: ht

Exception ignored in: <function _xla_gc_callback at 0x78247b5f4f40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/lib/__init__.py", line 127, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 


==((====))==  Unsloth 2025.12.6: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
==((====))==  Unsloth 2025.12.6: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
------------------------------------------------------------------------------

KeyboardInterrupt: 

In [ ]:
# FULL TEST SET EVALUATION: BLEU & chrF FOR ALL 4 MODELS (RESUMABLE + LENGTH-BASED STOPPING + FIXED LENGTH BUG)

MODEL_DIR="/content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project" #@@@ CHANGE for to TA directory for grading

import sys
sys.setrecursionlimit(5000)

import json
import re
import torch
import gc
import numpy as np
from collections import Counter
from unsloth import FastLanguageModel
from peft import PeftModel
from transformers import StoppingCriteria, StoppingCriteriaList
from datasets import Dataset
import pandas as pd
from IPython.display import display, HTML
import os

print("=== FULL TEST SET EVALUATION: BLEU-4 & chrF FOR ALL 4 MODELS (RESUMABLE) ===")

# Load full test set
test_path = "/content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/splits/test_paired.jsonl"
test_lines = [json.loads(line) for line in open(test_path, "r", encoding="utf-8")]
test_dataset = Dataset.from_list(test_lines)

print(f"Full test set size: {len(test_dataset)} examples\n")

# Extract sources and references ONCE
sources = [ex["en"] for ex in test_dataset]
references = [ex["ka"] for ex in test_dataset]

# All 4 models (order matters for resumption)
model_order = ["baseline", "mixed", "chron", "rev"]
models_to_eval = {
    "baseline": {"path": f"{MODEL_DIR}/baseline_sft_final", "display": "Baseline (direct SFT)"},
    "mixed":    {"path": f"{MODEL_DIR}/mixed_sft_final",    "display": "Mixed MLM + SFT"},
    "chron":    {"path": f"{MODEL_DIR}/chron_sft_final",    "display": "Chronological MLM + SFT"},
    "rev":      {"path": f"{MODEL_DIR}/rev_sft_final",      "display": "Reverse-Chronological MLM + SFT"},
}

# Prompt template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Translate the following English text to Georgian.

### Input:
{en}

### Response:
{}"""

# Length + punctuation stopping criteria
class CombinedLengthStop(StoppingCriteria):
    def __init__(self, tokenizer, prompt_text, lower_limit, upper_limit):
        self.tokenizer = tokenizer
        self.prompt_text = prompt_text
        self.lower_limit = lower_limit
        self.upper_limit = upper_limit
        self.punct = [".", "!", "?", "჻", "\n\n", "###"]

    def __call__(self, input_ids, scores, **kwargs):
        decoded = self.tokenizer.decode(input_ids[0], skip_special_tokens=True)
        generated = decoded[len(self.prompt_text):].strip()
        gen_len = len(generated)
        ends_with_punct = any(generated.endswith(p) for p in self.punct)

        if gen_len > self.upper_limit:
            return True
        if ends_with_punct and gen_len >= self.lower_limit:
            return True
        return False

# Storage for predictions
predictions = {name: [] for name in model_order}

# Track which models are already completed via checkpoints
completed_models = []

# Check for existing checkpoints and load them
for name in model_order:
    checkpoint_path = f"{MODEL_DIR}/full_test_predictions_up_to_{name}.csv"
    if os.path.exists(checkpoint_path):
        print(f"Found checkpoint: {checkpoint_path} → Loading predictions for models up to '{name}'")
        df = pd.read_csv(checkpoint_path)
        for col in model_order:
            if col in df.columns:
                predictions[col] = df[col].tolist()
                if col not in completed_models:
                    completed_models.append(col)
        print(f"Loaded: {', '.join(completed_models)}\n")
    else:
        # Stop checking further if one is missing (assumes sequential saving)
        break

# Generate only for remaining models
for name in model_order:
    if name in completed_models:
        print(f"Skipping {models_to_eval[name]['display']} (already completed and loaded from checkpoint)\n")
        continue

    info = models_to_eval[name]
    display_name = info["display"]
    checkpoint_path = info["path"]

    print(f"Loading {display_name} from: {checkpoint_path}")

    base_model, tokenizer = FastLanguageModel.from_pretrained(
        "unsloth/Llama-3.2-3B-Instruct",
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )

    model = PeftModel.from_pretrained(base_model, checkpoint_path)
    FastLanguageModel.for_inference(model)

    print(f"Generating predictions for {display_name}...\n")

    for idx, example in enumerate(test_dataset):
        src = example["en"]

        src_len = len(src)
        lower = int(src_len * 0.9)
        upper = int(src_len * 1.1)

        prompt = alpaca_prompt.format(en=src)

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        output = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.0,
            do_sample=False,
            use_cache=True,
            eos_token_id=None,
            pad_token_id=tokenizer.eos_token_id,
            stopping_criteria=StoppingCriteriaList([CombinedLengthStop(tokenizer, prompt, lower, upper)]),
        )

        raw = tokenizer.decode(output[0], skip_special_tokens=True)

        if "### Response:" in raw:
            translation = raw.split("### Response:")[-1].strip()
        else:
            translation = raw[len(prompt):].strip()

        # Trim to first complete sentence
        translation = re.split(r'[.!?჻]', translation)[0].strip()
        if translation and translation[-1] not in ".!?჻":
            translation += "."

        predictions[name].append(translation)

        if (idx + 1) % 50 == 0:
            print(f"  {idx+1}/{len(test_dataset)} completed for {display_name}")

        del inputs, output
        torch.cuda.empty_cache()

    # === SAVE CHECKPOINT AFTER THIS MODEL ===
    completed_models.append(name)

    temp_data = {"source_en": sources, "reference_ka": references}
    for comp_name in completed_models:
        temp_data[comp_name] = predictions[comp_name]

    temp_df = pd.DataFrame(temp_data)
    intermediate_csv = f"{MODEL_DIR}/full_test_predictions_up_to_{name}.csv"
    temp_df.to_csv(intermediate_csv, index=False, encoding="utf-8")
    print(f"✓ Checkpoint saved: {intermediate_csv} (includes {', '.join(completed_models)})\n")

    del model, base_model
    gc.collect()
    torch.cuda.empty_cache()

    print(f"{display_name} predictions complete\n")

# === METRIC COMPUTATION (same as before) ===
def compute_bleu(ref, pred, max_n=4):
    ref_words = ref.split()
    pred_words = pred.split()
    if len(pred_words) == 0:
        return 0.0
    brev = min(1.0, len(pred_words) / len(ref_words)) if len(ref_words) > 0 else 0.0

    precisions = []
    for n in range(1, max_n+1):
        ref_ng = Counter(tuple(ref_words[i:i+n]) for i in range(len(ref_words)-n+1))
        pred_ng = Counter(tuple(pred_words[i:i+n]) for i in range(len(pred_words)-n+1))
        correct = sum(min(pred_ng[g], ref_ng.get(g, 0)) for g in pred_ng)
        total = sum(pred_ng.values())
        p = correct / total if total > 0 else 0.0
        precisions.append(p)

    if min(precisions) == 0:
        return 0.0
    geo_mean = np.exp(np.mean(np.log(precisions)))
    return brev * geo_mean

def compute_chrf(ref, pred, n=6, beta=2):
    def char_ngrams(s, k):
        return [s[i:i+k] for i in range(len(s)-k+1)]

    matches = [0] * (n+1)
    ref_total = [0] * (n+1)
    pred_total = [0] * (n+1)

    for k in range(1, n+1):
        ref_ng = Counter(char_ngrams(ref, k))
        pred_ng = Counter(char_ngrams(pred, k))
        matches[k] = sum(min(pred_ng[g], ref_ng.get(g, 0)) for g in pred_ng)
        ref_total[k] = sum(ref_ng.values())
        pred_total[k] = sum(pred_ng.values())

    precision = np.mean([matches[k] / pred_total[k] if pred_total[k] > 0 else 0 for k in range(1, n+1)])
    recall = np.mean([matches[k] / ref_total[k] if ref_total[k] > 0 else 0 for k in range(1, n+1)])
    if precision + recall == 0:
        return 0.0
    f = (1 + beta**2) * precision * recall / (beta**2 * precision + recall)
    return f

# Compute and display scores
print("Computing BLEU-4 and chrF scores...\n")
scores = {}
for name in model_order:
    bleu_scores = [compute_bleu(ref, pred) for ref, pred in zip(references, predictions[name])]
    chrf_scores = [compute_chrf(ref, pred) for ref, pred in zip(references, predictions[name])]

    avg_bleu = np.mean(bleu_scores)
    avg_chrf = np.mean(chrf_scores)

    scores[name] = {"BLEU-4": avg_bleu, "chrF": avg_chrf}
    print(f"{models_to_eval[name]['display']}:")
    print(f"  Avg BLEU-4: {avg_bleu:.4f}")
    print(f"  Avg chrF:   {avg_chrf:.4f}\n")

# Final saves
data = {"source_en": sources, "reference_ka": references}
for name in model_order:
    data[name] = predictions[name]

df_pred = pd.DataFrame(data)
pred_csv = f"{MODEL_DIR}/full_test_predictions_all_models.csv"
df_pred.to_csv(pred_csv, index=False, encoding="utf-8")
print(f"Full predictions saved: {pred_csv}")

df_scores = pd.DataFrame(scores).T
scores_csv = f"{MODEL_DIR}/full_test_scores_summary.csv"
df_scores.to_csv(scores_csv)
print(f"Scores summary saved: {scores_csv}")

display(HTML(df_scores.to_html(float_format=lambda x: f"{x:.4f}")))

print("\nEvaluation complete! Compare the four models to determine whether chronological or reverse-chronological continued pretraining on historical Georgian data improves English → Modern Georgian translation performance over mixed-order or no historical pretraining.")

=== FULL TEST SET EVALUATION: SPECIFIC PROMPT (GEORGIAN SCRIPT ONLY) ===
Full test set size: 5000 examples

Loading Baseline (direct SFT) from: /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/baseline_sft_final
==((====))==  Unsloth 2025.12.5: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


ValueError: Can't find 'adapter_config.json' at '/content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/baseline_sft_final'

In [6]:
# COMET EVALUATION FOR ALL 4 MODELS ON FULL TEST SET PREDICTIONS

!pip install unbabel-comet

# If you encounter GPU memory issues with the default large model, you can switch to a smaller one later in the code.

import pandas as pd
import numpy as np
from IPython.display import display, HTML
from comet import download_model, load_from_checkpoint

print("=== COMET EVALUATION FOR ALL 4 MODELS ===")

# Paths (same as before – adjust if needed)
MODEL_DIR = "/content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/Turn-in-Folder"  #@@@ UPDATE IF YOUR PATH DIFFERS
pred_csv = f"{MODEL_DIR}/full_test_predictions_all_models.csv"

# Load the full predictions DataFrame
df = pd.read_csv(pred_csv)

print(f"Loaded predictions: {len(df)} examples\n")
print("Columns:", list(df.columns))

# Required columns
sources = df["source_en"].tolist()
references = df["reference_ka"].tolist()

# All 4 model columns
models_to_eval = {
    "baseline": "Baseline (direct SFT)",
    "mixed":    "Mixed MLM + SFT",
    "chron":    "Chronological MLM + SFT",
    "rev":      "Reverse-Chronological MLM + SFT",
}

# Choose COMET model
comet_model_name = "Unbabel/wmt22-comet-da"
# comet_model_name = "eamt22-cometinho-da"  # Uncomment for smaller model

print(f"Downloading and loading COMET model: {comet_model_name}")
model_path = download_model(comet_model_name)
comet_model = load_from_checkpoint(model_path)

# Compute COMET scores for each system
print("\nComputing COMET scores...\n")
comet_scores = {}

for col_name, display_name in models_to_eval.items():
    hypotheses = df[col_name].tolist()

    # Prepare data in the format expected by COMET
    data = [
        {"src": src, "mt": hyp, "ref": ref}
        for src, hyp, ref in zip(sources, hypotheses, references)
    ]

    # Predict segment-level scores
    seg_scores = comet_model.predict(data, batch_size=32, gpus=1)  # Adjust batch_size if memory issues
    mean_score = np.mean(seg_scores.scores)

    comet_scores[col_name] = {"COMET": mean_score}

    print(f"{display_name}:")
    print(f"  Avg COMET: {mean_score:.4f}\n")

# Save COMET results alongside previous metrics (optional merge)
# First load previous scores if they exist
scores_csv = f"{MODEL_DIR}/full_test_scores_summary.csv"
try:
    prev_scores_df = pd.read_csv(scores_csv, index_col=0)
except FileNotFoundError:
    prev_scores_df = pd.DataFrame()

# Create new COMET DataFrame
comet_df = pd.DataFrame(comet_scores).T

# Merge with previous (BLEU-4 & chrF)
combined_df = pd.concat([prev_scores_df, comet_df], axis=1)

# Save updated summary
combined_csv = f"{MODEL_DIR}/full_test_scores_summary_with_comet.csv"
combined_df.to_csv(combined_csv)

print(f"Updated scores summary (including COMET) saved: {combined_csv}")

# Display nicely
display(HTML(combined_df.to_html(float_format=lambda x: f"{x:.4f}")))

print("\nCOMET evaluation complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 49.5 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not curren

=== COMET EVALUATION FOR ALL 4 MODELS ===
Loaded predictions: 5000 examples

Columns: ['source_en', 'reference_ka', 'baseline', 'mixed', 'chron', 'rev']


.gitattributes: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Chronological MLM + SFT:
  Avg COMET: 0.3631



Predicting DataLoader 0: 100%|██████████| 157/157 [00:35<00:00,  4.39it/s]


Reverse-Chronological MLM + SFT:
  Avg COMET: 0.3476

Updated scores summary (including COMET) saved: /content/drive/MyDrive/Colab_Notebooks/CS_479/Final_Project/Turn-in-Folder/full_test_scores_summary_with_comet.csv


,COMET
baseline,0.3006
mixed,0.3454
chron,0.3631
rev,0.3476



COMET evaluation complete!
COMET is a neural metric with higher correlation to human judgments than BLEU or chrF.
Higher scores (closer to 1.0) indicate better translation quality.
Use these COMET scores together with BLEU-4 and chrF to robustly compare your four models
and test your hypothesis: does chronological or reverse-chronological continued pretraining
yield higher-quality English → Modern Georgian translations than mixed order or no historical data?
